# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


In [ ]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [ ]:
asia = df[df['Region'] == 'Asia']
highlight = 'China'

fig = go.Figure()

for country in asia['Country'].unique():
    cdata = asia[asia['Country'] == country]
    if country == highlight:
        fig.add_trace(go.Scatter(x=cdata['Year'], y=cdata['CO2_Mt'], mode='lines',
                                 line=dict(color='#2E75B6', width=3), name=country, showlegend=False))
    else:
        fig.add_trace(go.Scatter(x=cdata['Year'], y=cdata['CO2_Mt'], mode='lines',
                                 line=dict(color='#DDDDDD', width=1.5), name=country, showlegend=False))

last = asia[(asia['Country'] == highlight) & (asia['Year'] == asia['Year'].max())]
fig.add_annotation(x=last['Year'].values[0], y=last['CO2_Mt'].values[0],
                   text=f'<b>{highlight}</b>', showarrow=False, xanchor='left', xshift=6,
                   font=dict(color='#2E75B6', size=12, family='Arial'))

fig.update_layout(
    title="China's CO2 emissions tripled since 2000 — dwarfing all other Asian economies",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=False, title=''),
    yaxis=dict(gridcolor='#EEEEEE', gridwidth=1, title='CO2 Emissions (Mt)'),
    margin=dict(l=60, r=80, t=55, b=40), height=500
)

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [ ]:
regional = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
sg = regional[regional['Year'].isin([2000, 2022])].copy()

changes = sg.pivot(index='Region', columns='Year', values='CO2_Mt')
changes['increased'] = changes[2022] > changes[2000]

fig = go.Figure()

for region in changes.index:
    row = changes.loc[region]
    color = '#C0392B' if row['increased'] else '#27AE60'
    rdata = sg[sg['Region'] == region].sort_values('Year')

    fig.add_trace(go.Scatter(
        x=rdata['Year'].astype(str), y=rdata['CO2_Mt'], mode='lines+markers+text',
        line=dict(color=color, width=2.5), marker=dict(size=8, color=color),
        text=[f'{row[2000]:.0f}  ', f'  {region}<br>  {row[2022]:.0f}'],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, color=color, family='Arial'),
        showlegend=False
    ))

fig.update_layout(
    title='Asia surged while Europe and North America cut emissions — 2000 vs 2022 regional averages',
    xaxis=dict(tickvals=['2000', '2022'], showgrid=False, range=[-0.5, 1.5]),
    yaxis=dict(showgrid=False, showticklabels=False, title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=100, r=140, t=55, b=40), height=600, width=900
)

fig.show()